In [43]:
import kagglehub
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from transformers import pipelines, Trainer, TrainingArguments
from transformers import ViTImageProcessor, ViTForImageClassification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from torch.utils.data import Dataset # A Blueprint form Pytoch, explaining standard way of feeding data to a model
import torch # used to convert labels into a format the models understand 
from PIL import Image

Personal Notes: you will need a way to encode the test data and convert the labels as well. So we are transforming both x and y for the machine to understand

In [44]:
checkpoint = "google/vit-base-patch16-224"
processor = ViTImageProcessor.from_pretrained(checkpoint)

In [45]:
label2id = {"akiec":0, "bcc":1, "bkl":2, "df":3, "mel":4, "nv":5, "vasc":6}
id2label = {value:key for key,value in label2id.items()}

## Download the datset into the .cache directory

In [46]:
# Download latest version
path = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")
print("Path to dataset files:", path)

Path to dataset files: C:\Users\tle72\.cache\kagglehub\datasets\kmader\skin-cancer-mnist-ham10000\versions\2


In [47]:
#stores everything in the .cache folder, not project folder
print(os.listdir(path))

['HAM10000_images_part_1', 'HAM10000_images_part_2', 'HAM10000_metadata.csv', 'hmnist_28_28_L.csv', 'hmnist_28_28_RGB.csv', 'hmnist_8_8_L.csv', 'hmnist_8_8_RGB.csv']


## Prepare the dataset

In [48]:
csv_path = os.path.join(path, "HAM10000_metadata.csv")
data = pd.read_csv(csv_path)
data.head()

,lesion_id,image_id,dx,dx_type,age,sex,localization
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear


In [49]:
# needed to initilize path in the beginning to ensure the we are searching for the data within the .cache, and not just the current directory
image_dir1 = os.path.join(path, "HAM10000_images_part_1")
image_dir2 = os.path.join(path, "HAM10000_images_part_2")

In [50]:
def get_image_path(image_id):
    for i in [image_dir1, image_dir2]:   # loop through 2 folders at once
        p = os.path.join(i, image_id + ".jpg")
        if os.path.exists(p):
            return p
    return None

In [59]:
class encoder(Dataset):  #encoder class inherits from Pytorch.Dataset

    #initilization class
    def __init__(self, df, processor, label2id): 
        self.df = df.reset_index(drop=True) # When we retireve data set, we ensure all indexes are 0,1,2,... and not jumping 3,7,10,... 
        self.processor = processor
        self.label2id = label2id

    # len() function should also work for this object    
    def __len__(self):
        return len(self.df) 

    def __getitem__(self, idx):
        #Retreieve image_title, based on the of the row_id --> use image title to retrieve image file 
        row = self.df.iloc[idx]
        img_path = get_image_path(row["image_id"])
        images = Image.open(img_path).convert("RGB")
        
        #Encoding the testing data 
        encodings = self.processor(images = images, return_tensors = "pt") # Image --> pytorch tensor
        encodings = {key:value.squeeze(0)   for key,value in encodings.items()} # flatten the tensor into 1 dimension 

        # labels (Stings) --> labels (labels)
        encodings["labels"] = torch.tensor(self.label2id[row["dx"]])

        #encodings should have image title, tensor_encodings, and it's assigned labels, ready to be inputed into model 
        return encodings 


#For Reference: image_id = data.iloc[0]["image_id"]+".jpg"
#accessing in row => data.iloc[0]
#accessing in columns => ["image_id"]

## Import the processor and model

In [60]:
train_df, valid_df = train_test_split(data, test_size = 0.2, stratify = data["dx"]) #split the data but keep the same class distribution. 

In [61]:
train_data = encoder(train_df, processor, label2id)
valid_data = encoder(valid_df, processor, label2id)

In [62]:
model =  ViTForImageClassification.from_pretrained(checkpoint,
                                                   num_labels = 7,
                                                   id2label = id2label,
                                                   label2id = label2id,
                                                   ignore_mismatched_sizes = True
                                                  )
# fix later so it doesn ignore the mismatched sizes.
# Also reduce the number of classes

Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([7]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([7, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [63]:
train_arguments = TrainingArguments(output_dir = "saved_models/BaselineModel", eval_strategy = "epoch")

In [64]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [65]:
trainer = Trainer(model = model,
                  args = train_arguments,
                  train_dataset = train_data,
                  eval_dataset = valid_data,
                  compute_metrics = compute_metrics)

In [66]:
trainer.train()

C:\Users\tle72\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

## Extra Code for Decoding

In [ ]:
## Setting up the directory of os
os.chdir("C:/Users/tle72/.cache/kagglehub/datasets/kmader/skin-cancer-mnist-ham10000/versions/2")

In [ ]:
# get full path of the first image
image_id = data.iloc[0]["image_id"]+".jpg"
fullpath = os.path.join(image_dir1, image_id)
print(fullpath)
"""
image = Image.open(fullpath)
plt.imshow(image)
plt.axis("off")
plt.show()
"""

In [ ]:
print(os.getcwd())